In [ ]:
import json
from pathlib import Path

import pandas as pd


def resolve_repo_root() -> Path:
    """Support running the notebook from repo root or from notebooks/ as CWD."""
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent]
    target_rel = Path('data/NUST Bank-Product-Knowledge.xlsx')

    for candidate in candidates:
        if (candidate / target_rel).exists():
            return candidate

    raise FileNotFoundError(
        'Could not locate data/NUST Bank-Product-Knowledge.xlsx from current working directory.'
    )


repo_root = resolve_repo_root()
workbook_path = repo_root / 'data' / 'NUST Bank-Product-Knowledge.xlsx'
out_dir = repo_root / 'data' / 'processed'
out_path = out_dir / 'cleaned_bank_knowledge.json'

excluded_sheets = {'Main', 'Rate Sheet July 1 2024'}

xls = pd.ExcelFile(workbook_path)
records = []

for sheet_name in xls.sheet_names:
    if sheet_name in excluded_sheets:
        continue

    df = pd.read_excel(workbook_path, sheet_name=sheet_name, header=None)

    current_question = None
    current_answer_parts = []

    for _, row in df.iterrows():
        non_empty_values = [str(v).strip() for v in row.dropna().tolist() if str(v).strip()]
        if not non_empty_values:
            continue

        row_text = ' '.join(non_empty_values).strip()

        if '?' in row_text:
            if current_question is not None:
                answer_text = ' '.join(current_answer_parts).strip()
                records.append(
                    {
                        'sheet': sheet_name,
                        'topic': sheet_name,
                        'question': current_question,
                        'answer': answer_text,
                    }
                )

            current_question = row_text
            current_answer_parts = []
        else:
            if current_question is not None:
                current_answer_parts.append(row_text)

    if current_question is not None:
        answer_text = ' '.join(current_answer_parts).strip()
        records.append(
            {
                'sheet': sheet_name,
                'topic': sheet_name,
                'question': current_question,
                'answer': answer_text,
            }
        )

out_dir.mkdir(parents=True, exist_ok=True)
out_path.write_text(json.dumps(records, ensure_ascii=False, indent=2), encoding='utf-8')

print(f'Total records: {len(records)}')
print(f'Output saved to: {out_path}')



In [1]:
import pandas as pd
import json
import os

# Define paths
excel_path = "../data/NUST Bank-Product-Knowledge.xlsx"
output_path = "../data/processed/rate_sheet_knowledge.json"

def process_rate_sheet(path):
    # Load only the Rate Sheet tab
    df = pd.read_excel(path, sheet_name="Rate Sheet July 1 2024", header=None)
    
    rate_data = []
    current_product = None
    
    for index, row in df.iterrows():
        # Clean the row
        clean_row = [str(c).strip() for c in row if pd.notna(c) and str(c).strip() != ""]
        
        if not clean_row:
            continue
            
        # Detect Product Headers (Usually the only thing in a row or underlined in blue in your SS)
        if len(clean_row) == 1 and not any(char.isdigit() for char in clean_row[0]):
            current_product = clean_row[0]
            continue
            
        # Detect Table Rows (Rows that have a payout/tenor and a percentage)
        if any("%" in cell or "." in cell for cell in clean_row) and current_product:
            # We stitch the product name to the specific rate details
            details = " | ".join(clean_row)
            rate_data.append({
                "sheet": "Rate Sheet",
                "topic": "Profit Rates",
                "question": f"What is the profit rate and payout for {current_product}?",
                "answer": f"For {current_product}, the details are: {details}"
            })
            
    return rate_data

# Execute and save
try:
    final_rates = process_rate_sheet(excel_path)
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(final_rates, f, indent=2)
    print(f"Successfully processed {len(final_rates)} rate entries into {output_path}")
except Exception as e:
    print(f"Error: {e}")

Successfully processed 31 rate entries into ../data/processed/rate_sheet_knowledge.json


In [5]:
import json

# Load both files with explicit UTF-8 encoding
try:
    with open("../data/processed/cleaned_bank_knowledge.json", 'r', encoding='utf-8') as f:
        qa_data = json.load(f)
        
    with open("../data/processed/rate_sheet_knowledge.json", 'r', encoding='utf-8') as f:
        rate_data = json.load(f)

    # Combine the lists
    total_knowledge = qa_data + rate_data

    # Save as the final master file
    with open("../data/processed/master_bank_knowledge.json", 'w', encoding='utf-8') as f:
        json.dump(total_knowledge, f, indent=2, ensure_ascii=False)

    print(f"Success! Master knowledge base created with {len(total_knowledge)} total entries.")

except UnicodeDecodeError as e:
    print(f"Encoding Error: {e}. Check if the files were saved correctly in UTF-8.")
except FileNotFoundError as e:
    print(f"File Error: {e}. Make sure the JSON files exist in data/processed/.")

Success! Master knowledge base created with 324 total entries.


In [8]:
import json
import os

# Paths
faq_input_path = "../data/funds_transfer_app_features_faq.json"
faq_output_path = "../data/processed/mobile_app_knowledge.json"

def process_mobile_faqs(path):
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    faq_data = []
    for category_item in data['categories']:
        category_name = category_item['category']
        for q_item in category_item['questions']:
            faq_data.append({
                "sheet": "Mobile App",
                "topic": category_name,
                "question": q_item['question'].strip(),
                "answer": q_item['answer'].strip()
            })
    return faq_data

# Run and Save
mobile_knowledge = process_mobile_faqs(faq_input_path)
with open(faq_output_path, 'w', encoding='utf-8') as f:
    json.dump(mobile_knowledge, f, indent=2, ensure_ascii=False)

print(f"Processed {len(mobile_knowledge)} Mobile App FAQs.")

Processed 15 Mobile App FAQs.


In [9]:
import json
import re

def clean_text(text):
    if not text: return ""
    # 1. Remove stray bullet points and special symbols
    text = re.sub(r'[•◦○o\*·◦]', '', text) 
    # 2. Fix multiple spaces left behind by bullet removal
    text = re.sub(r'\s+', ' ', text)
    # 3. Trim whitespace
    return text.strip()

# 1. Load all three processed files
files_to_merge = [
    "../data/processed/cleaned_bank_knowledge.json",
    "../data/processed/rate_sheet_knowledge.json",
    "../data/processed/mobile_app_knowledge.json"
]

master_knowledge = []

for file_path in files_to_merge:
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
        master_knowledge.extend(data)

# 2. Apply Normalization to every entry
normalized_knowledge = []
for entry in master_knowledge:
    normalized_entry = {
        "sheet": entry['sheet'],
        "topic": entry['topic'],
        "question": clean_text(entry['question']),
        "answer": clean_text(entry['answer'])
    }
    normalized_knowledge.append(normalized_entry)

# 3. Save the final "Master" dataset for the Vector Database
master_output_path = "../data/processed/master_bank_knowledge.json"
with open(master_output_path, 'w', encoding='utf-8') as f:
    json.dump(normalized_knowledge, f, indent=2, ensure_ascii=False)

print(f"Normalization complete! Total records: {len(normalized_knowledge)}")
print(f"Final dataset saved to: {master_output_path}")

Normalization complete! Total records: 339
Final dataset saved to: ../data/processed/master_bank_knowledge.json


In [10]:
import pandas as pd

# Load your final normalized base
df_audit = pd.read_json("../data/processed/master_bank_knowledge.json")

print("--- Data Integrity Report ---")
print(f"Total Records: {len(df_audit)}")
print(f"Unique Sources (Sheets): {df_audit['sheet'].nunique()}")

# Check for empty fields
null_counts = df_audit.isnull().sum()
empty_strings = (df_audit == "").sum()

print("\nMissing Values per Column:")
print(null_counts + empty_strings)

# Check for unusually short answers (potential parsing errors)
short_answers = df_audit[df_audit['answer'].str.len() < 10]
print(f"\nFlagged short answers: {len(short_answers)}")

--- Data Integrity Report ---
Total Records: 339
Unique Sources (Sheets): 35

Missing Values per Column:
sheet       0
topic       0
question    0
answer      0
dtype: int64

Flagged short answers: 15
